In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [3]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)   

In [4]:
X_train = pad_sequences(X_train, maxlen=100)
X_test = pad_sequences(X_test, maxlen=100)

In [5]:
model = Sequential()

model.add(Embedding(10000, 32, input_length=100))
model.add(SimpleRNN(5, return_sequences=True))
model.add(SimpleRNN(5))
model.add(Dense(1, activation='sigmoid'))

In [6]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 32)           320000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 100, 5)            190       
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 5)                 55        
                                                                 
 dense (Dense)               (None, 1)                 6         
                                                                 
Total params: 320,251
Trainable params: 320,251
Non-trainable params: 0
_________________________________________________________________


In [7]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [8]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test),
    batch_size=32
)

Epoch 1/5
782/782 [==============================] - 416s 524ms/step - loss: 0.5518 - accuracy: 0.7288 - val_loss: 0.4569 - val_accuracy: 0.8045
Epoch 2/5
782/782 [==============================] - 228s 292ms/step - loss: 0.4131 - accuracy: 0.8243 - val_loss: 0.4443 - val_accuracy: 0.8031
Epoch 3/5
782/782 [==============================] - 253s 323ms/step - loss: 0.3038 - accuracy: 0.8843 - val_loss: 0.4612 - val_accuracy: 0.8092
Epoch 4/5
782/782 [==============================] - 220s 281ms/step - loss: 0.2376 - accuracy: 0.9141 - val_loss: 0.5025 - val_accuracy: 0.8045
Epoch 5/5
782/782 [==============================] - 233s 297ms/step - loss: 0.1943 - accuracy: 0.9320 - val_loss: 0.5370 - val_accuracy: 0.8076


In [9]:
word_index = imdb.get_word_index()

1641221/1641221 [==============================] - 1s 0us/step


In [10]:
def encode_text(text):
    words = text.lower().split()
    
    encoded = []
    for word in words:
        if word in word_index:
            encoded.append(word_index[word] + 3)  # offset
        else:
            encoded.append(2)  # unknown token
    
    return encoded

In [11]:
def predict_review(text):
    encoded = encode_text(text)
    
    padded = pad_sequences([encoded], maxlen=100)
    
    prediction = model.predict(padded)
    
    if prediction[0][0] > 0.5:
        print("Positive Review 😊")
    else:
        print("Negative Review 😡")
    
    print("Confidence:", prediction[0][0])

In [15]:
predict_review("This movie is amazing and I loved it")

predict_review("not good but yes worst film ever made, very good")

1/1 [==============================] - 0s 50ms/step
Positive Review 😊
Confidence: 0.9788897
1/1 [==============================] - 0s 45ms/step
Negative Review 😡
Confidence: 0.24493621
